# Worksheet 4, Part 2 — Stellar parameters
### HE 1002-0755

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from astropy.table import Table
os.makedirs("results", exist_ok=True); os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(42)
N = 10**5

plt.rcParams.update({"font.family": "serif", "font.size": 11, "xtick.direction": "in",
    "ytick.direction": "in", "xtick.top": True, "ytick.right": True, "legend.frameon": False})

## 1. Inputs

In [ ]:
PHOT = "../../../data/photometry"
uc = Table.read(f"{PHOT}/322A_he1002.ecsv")[0]
tm = Table.read(f"{PHOT}/246_he1002.ecsv")[0]

Vmag, Vmag_error = float(uc["Vmag"]), float(uc["e_Vmag"]) / 100.    # UCAC4 e_Vmag is in centimag
Jmag, Jmag_error = float(tm["Jmag"]), float(tm["e_Jmag"])
Kmag, Kmag_error = float(tm["Kmag"]), float(tm["e_Kmag"])

color_excess, color_excess_error = 0.0451, 0.0018      # E(B-V), S&F 2011 mean, IRSA
AV_SandF = 0.139                                       # IRSA S&F table, CTIO V (used by the log g notebook)

feh, feh_error = -3.0, 0.3                             # PROVISIONAL: WS2 summary plot; replace with SMHR Fe I

g = Table.read("../../../data/gaia/he1002_gaia.ecsv")[0]
parallax, parallax_error = float(g["parallax"]), float(g["parallax_error"])
post = json.load(open("../../ws2/part2_observation_details/results/distance_posterior.json"))
zp = post["zero_point_mas"]

print(f"V = {Vmag:.3f} ± {Vmag_error:.3f}   J = {Jmag:.3f} ± {Jmag_error:.3f}   Ks = {Kmag:.3f} ± {Kmag_error:.3f}")
print(f"E(B-V) = {color_excess} ± {color_excess_error}")
print(f"parallax = {parallax:.4f} ± {parallax_error:.4f} mas  (Lindegren zero-point {zp:+.4f} mas)")
print(f"[Fe/H] = {feh} ± {feh_error}  (provisional)")

## 2.1 Effective temperature from colours

In [ ]:
E  = color_excess + color_excess_error * rng.normal(size=N)
V  = Vmag + Vmag_error * rng.normal(size=N)
J  = Jmag + Jmag_error * rng.normal(size=N)
K  = Kmag + Kmag_error * rng.normal(size=N)
FE = feh + feh_error * rng.normal(size=N)

V0, J0, K0 = V - 3.315*E, J - 0.902*E, K - 0.367*E

def theta_VJ(c, f): return 0.4669 + 0.3849*c - 0.0350*c**2 - 0.0140*c*f + 0.0225*f + 0.0011*f**2
def theta_VK(c, f): return 0.5057 + 0.2600*c - 0.0146*c**2 - 0.0131*c*f + 0.0288*f + 0.0016*f**2

Teff_VK = 5040 / theta_VK(V0 - K0, FE)
Teff_VJ = 5040 / theta_VJ(V0 - J0, FE)

def pct(x): lo, med, hi = np.percentile(x, [16, 50, 84]); return med, hi - med, med - lo

print(f"(V-Ks)_0 = {np.median(V0-K0):.3f}   (V-J)_0 = {np.median(V0-J0):.3f}")
for name, T in (("V-Ks", Teff_VK), ("V-J ", Teff_VJ)):
    m, p, n = pct(T); print(f"Teff({name}) = {m:.0f} +{p:.0f} -{n:.0f} K")

In [ ]:
for f in (-2.7, -3.0, -3.3):
    print(f"[Fe/H] = {f:+.1f}:  Teff(V-Ks) = {np.median(5040/theta_VK(V0-K0, f)):.0f} K")
Teff, Teff_err = float(np.median(Teff_VK)), float(np.std(Teff_VK))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.hist(Teff_VK, 80, histtype="step", lw=1.5, label=r"$V-K_s$")
ax.hist(Teff_VJ, 80, histtype="step", lw=1.5, label=r"$V-J$")
ax.axvline(Teff, color="k", lw=0.8)
ax.set_xlabel(r"$T_{\rm eff}$ (K)"); ax.set_ylabel("samples"); ax.legend()
fig.tight_layout(); fig.savefig("plots/teff_samples_HE1002-0755.pdf"); fig.savefig("plots/teff_samples_HE1002-0755.png", dpi=150)
print(f"Adopted Teff = {Teff:.0f} ± {Teff_err:.0f} K")

## 2.2 Surface gravity
### (1) Isochrone

In [ ]:
img = plt.imread("original/isochrones_ws4.png")
# plot frame in pixels: x 135..1395, y 8..1518; ticks 6500 K at x=263, 4000 K at x=1330; log g 0 at y=134, 5 at y=1392
kx = (1330-263)/2500.; ky = (1392-134)/5.
ext = [6500 + (263-135)/kx, 6500 - (1395-263)/kx, (1518-134)/ky, (8-134)/ky]
crop = img[8:1518, 135:1395]

# [Fe/H] = -3.0 RGB, read by eye from the figure
ISO_M30 = np.array([(5500, 3.40), (5300, 2.90), (5100, 2.40), (4900, 1.85), (4800, 1.55),
                    (4700, 1.25), (4600, 0.95), (4500, 0.65)])
Tmin = ISO_M30[:, 0].min()
if Teff >= Tmin:
    logg_iso = float(np.interp(Teff, ISO_M30[::-1, 0], ISO_M30[::-1, 1]))
    iso_note = "on the track"
else:
    # cooler than the end of the [Fe/H] = -3.0 track in the figure: extend the upper-RGB slope linearly
    slope = (ISO_M30[-2, 1] - ISO_M30[-1, 1]) / (ISO_M30[-2, 0] - ISO_M30[-1, 0])
    logg_iso = float(ISO_M30[-1, 1] + slope*(Teff - ISO_M30[-1, 0]))
    iso_note = f"EXTRAPOLATED {Tmin - Teff:.0f} K beyond the cool end of the [Fe/H] = -3.0 track"
logg_iso_err = float(np.hypot(0.15, 0.003*Teff_err))

fig, ax = plt.subplots(figsize=(5, 6))
ax.imshow(crop, extent=ext, aspect="auto")
ax.plot(ISO_M30[:, 0], ISO_M30[:, 1], "r+", ms=9, mew=1.5, label="by-eye reading of [Fe/H] = -3.0")
ax.errorbar(Teff, logg_iso, xerr=Teff_err, yerr=logg_iso_err, fmt="*", ms=14, color="C0", mec="k", label="HE 1002-0755")
ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_xlabel(r"$T_{\rm eff}$ (K)"); ax.set_ylabel(r"$\log g$")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); fig.savefig("plots/isochrone_HE1002-0755.pdf"); fig.savefig("plots/isochrone_HE1002-0755.png", dpi=150)
print(f"log g (isochrone, [Fe/H] = -3.0) = {logg_iso:.2f} ± {logg_iso_err:.2f}   [{iso_note}]")

### (2) From the parallax

In [ ]:
mass = 0.8
T  = Teff + Teff_err * rng.normal(size=N)
V0s = (Vmag + Vmag_error*rng.normal(size=N)) - AV_SandF
FE = feh + feh_error * rng.normal(size=N)

def BC_alonso(T, f):
    x = np.log10(T) - 3.52
    return -0.0993/x + 0.02887 + 2.275*x - 4.425*x**2 + 0.3505*x*f - 0.05558*f - 0.005375*f**2

def logg_from_dm(dm):     # dm = 5 log10(d_pc) - 5, the distance modulus
    Mbol = V0s + BC_alonso(T, FE) - dm
    return 4.44 + np.log10(mass) + 4*np.log10(T/5772.) + 0.4*(Mbol - 4.74), Mbol

out = {}
pA = parallax + parallax_error*rng.normal(size=N)
pB = pA - zp
for key, p in (("A_raw_parallax", pA), ("B_zero_point_corrected", pB)):
    ok = p > 0
    lg, Mb = logg_from_dm(np.where(ok, -5*np.log10(np.where(ok, p, 1)*1e-3) - 5, np.nan))
    out[key] = dict(logg=pct(lg[ok]), Mbol=pct(Mb[ok]), frac_negative_parallax=float((~ok).mean()))

# C: sample the distance from the posterior percentiles (split normal around the median)
lo, med, hi = post["posteriors"]["Halo only"]["r_kpc"]
u = rng.normal(size=N); r = np.where(u < 0, med + u*(med-lo), med + u*(hi-med))
r = r[r > 0.5]
lg, Mb = logg_from_dm(5*np.log10(r[:N]*1e3) - 5 if r.size >= N else 5*np.log10(np.resize(r, N)*1e3) - 5)
out["C_bayesian_halo_prior"] = dict(logg=pct(lg), Mbol=pct(Mb), r_kpc=[lo, med, hi])

print(f"BC (Alonso 1999) = {np.median(BC_alonso(T, FE)):+.3f}")
for k, v in out.items():
    m, p, n = v["logg"]; mb = v["Mbol"][0]
    extra = f"   ({100*v['frac_negative_parallax']:.1f}% of samples have parallax <= 0)" if "frac_negative_parallax" in v else ""
    print(f"{k:24s} log g = {m:.2f} +{p:.2f} -{n:.2f}   Mbol = {mb:+.2f}{extra}")
logg_plx, logg_plx_p, logg_plx_m = out["C_bayesian_halo_prior"]["logg"]

In [ ]:
print(f"isochrone {logg_iso:.2f}   parallax (C) {logg_plx:.2f}   difference {logg_plx - logg_iso:+.2f} dex")
logg = logg_plx
iso_ok = Teff >= Tmin
logg_sys = abs(logg_plx - logg_iso) if iso_ok else None
if not iso_ok:
    print("The isochrone value is an extrapolation, so this difference is not a usable systematic error.")

In [ ]:
Tc = Teff
for i in range(20):
    Ts = Tc + 0*T                                    # fixed temperature, other inputs still sampled
    x = np.log10(Ts) - 3.52
    BCc = -0.0993/x + 0.02887 + 2.275*x - 4.425*x**2 + 0.3505*x*FE - 0.05558*FE - 0.005375*FE**2
    dm = 5*np.log10(np.resize(r, N)*1e3) - 5
    lg_c = np.median(4.44 + np.log10(mass) + 4*np.log10(Ts/5772.) + 0.4*(V0s + BCc - dm - 4.74))
    Tn = float(np.interp(lg_c, ISO_M30[:, 1][::-1], ISO_M30[:, 0][::-1]))
    if abs(Tn - Tc) < 1: break
    Tc = Tn
print(f"converged after {i+1} steps: Teff = {Tc:.0f} K, log g = {lg_c:.2f} lies on the [Fe/H] = -3.0 track")
print(f"compare: colour Teff {Teff:.0f} K;  WS2 Balmer-wing comparison ~4600-4650 K;  SMHR default 4500 K")
Teff_iso_consistent, logg_iso_consistent = Tc, float(lg_c)

## 2.3 Microturbulence

In [ ]:
def vt_rpa(lg): return 0.060*lg**2 - 0.569*lg + 2.585
lgs = logg_plx + 0.5*(logg_plx_p + logg_plx_m)*rng.normal(size=N)
vmic, vmic_err = float(np.median(vt_rpa(lgs))), float(np.std(vt_rpa(lgs)))
print(f"vmic = {vmic:.2f} ± {vmic_err:.2f} km/s at log g = {logg_plx:.2f}")
print(f"(at the isochrone log g {logg_iso:.2f} it would be {vt_rpa(logg_iso):.2f} km/s)")

fig, ax = plt.subplots(figsize=(6, 4))
img = plt.imread("original/vmic_logg_RPA_ws4.png")
x = np.linspace(-0.2, 5, 200)
ax.plot(x, vt_rpa(x), color="C0", lw=2, label="RPA fit (WS4)")
ax.errorbar(logg_plx, vmic, xerr=[[logg_plx_m], [logg_plx_p]], yerr=vmic_err, fmt="*", ms=14, color="C3", mec="k", label="HE 1002-0755")
ax.set_xlabel(r"$\log g$"); ax.set_ylabel(r"$v_{\rm mic}$ (km s$^{-1}$)"); ax.set_xlim(-0.5, 5); ax.set_ylim(0.5, 3.5); ax.legend()
fig.tight_layout(); fig.savefig("plots/vmic_HE1002-0755.pdf"); fig.savefig("plots/vmic_HE1002-0755.png", dpi=150)

## 3. Summary — values for the observations table

In [ ]:
res = dict(
    star="HE 1002-0755",
    status="PROVISIONAL: [Fe/H] input is the WS2 summary-plot estimate; re-run with SMHR Fe I from the group coadd",
    inputs=dict(V=Vmag, e_V=Vmag_error, J=Jmag, e_J=Jmag_error, Ks=Kmag, e_Ks=Kmag_error,
                EBV=color_excess, e_EBV=color_excess_error, A_V=AV_SandF, feh=feh, e_feh=feh_error,
                parallax_mas=parallax, e_parallax_mas=parallax_error, zero_point_mas=zp, mass_msun=mass),
    Teff_VKs=dict(zip(("med", "plus", "minus"), pct(Teff_VK))),
    Teff_VJ=dict(zip(("med", "plus", "minus"), pct(Teff_VJ))),
    logg_isochrone=dict(value=logg_iso, err=logg_iso_err),
    logg_parallax={k: v for k, v in out.items()},
    vmic=dict(value=vmic, err=vmic_err),
    isochrone_parallax_consistent=dict(Teff=Teff_iso_consistent, logg=logg_iso_consistent),
    adopted=dict(Teff=round(Teff), e_Teff=round(Teff_err), logg=round(logg, 2),
                 e_logg_stat=round(0.5*(logg_plx_p+logg_plx_m), 2), e_logg_sys=(round(logg_sys, 2) if logg_sys is not None else None),
                 vmic=round(vmic, 2), feh_input=feh))
json.dump(res, open("results/stellar_parameters_HE1002-0755.json", "w"), indent=1, default=float)
a = res["adopted"]
print(f"Teff  = {a['Teff']} ± {a['e_Teff']} K                     (V-Ks, Casagrande et al. 2010)")
print(f"log g = {a['logg']} ± {a['e_logg_stat']} (stat); isochrone vs parallax: " + (f"± {a['e_logg_sys']}" if a['e_logg_sys'] is not None else "not usable (colour Teff is off the track)"))
print(f"check: the parallax gravity sits on the [Fe/H] = -3.0 isochrone at Teff = {Teff_iso_consistent:.0f} K, log g = {logg_iso_consistent:.2f}")
print(f"vmic  = {a['vmic']} km/s                       (RPA relation)")
print(f"[Fe/H] input = {a['feh_input']}  -> next: SMHR Fe I with these Teff/log g/vmic, then iterate")

## 4. Things to raise with AF / in class